### Backend

#### package & library

In [1]:
try:
    from openpyxl import load_workbook, Workbook
except:
    !pip install openpyxl
from datetime import datetime

import asyncio
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Alignment, PatternFill
from openpyxl.utils import get_column_letter, range_boundaries
import numpy as np
import re, time
from tqdm.notebook import tqdm

#### Format

In [2]:
def row_for_clarity(ws, color = "CCFFCC"):
    ws.append(["This is a row for clarity.", ""])
    for idx, cell in enumerate(ws[ws.max_row]): # dotted highlight for clarity
        if idx%2 == 0: cell.fill = PatternFill(start_color=color, fill_type="solid")
def align(ws, align = 'left'):
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(horizontal=align)
def estimate_visual_width(text, ptrue = False):
    if isinstance(text, datetime): return 11 # datetime always convert to smaller
    width = 0
    for char in str(text):
        if re.search(r'[A-Z]', char): width += 1.5
        elif re.search(r'[a-z\s]', char): width += 0.75
        else: width += 1
        if ptrue: print(char, width, end=", ")
    if isinstance(text, float): return min(width, 11) # excel store first 18 digits only & display first 11 digits by default
    return width
def auto_fit(ws):
    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)#; print(col[4].value, type(col[4].value)) # debug
        for cell in col: # visual width depends on character type
            if max_length < estimate_visual_width(cell.value): 
                max_length = estimate_visual_width(cell.value)
                #if col_letter == "B": print(cell, estimate_visual_width(cell.value, ptrue = True)) # debug
        ws.column_dimensions[col_letter].width = max_length#; print(int(max_length)) # debug

#### Functions

In [3]:
def NOx_clipboard(filepath = "BPPS_PI_data_NOx_Jul.xlsm", day = 7):
    start = time.time()
    
    wb_original = load_workbook(filepath, data_only=True, read_only=True); print(f"source xlsx loaded after {time.time()-start:.2f} sec")
    wb_new = Workbook(); active = wb_new.active
    ws_new = wb_new.create_sheet(title = "10 in 1")

    # NO, NO2 paste
    NOx_transfer(wb_original, ws_new, [0 ,10], (7, 9), 1, [(15, 16)])
    row_for_clarity(ws_new); print(f'headers transferred after {time.time()-start:.2f} sec')
    NOx_transfer(wb_original, ws_new, [0 ,10], (11, 11+24*day-1), 1, [(15, 16)])
    row_for_clarity(ws_new); print(f'NO, NO2 transferred after {time.time()-start:.2f} sec')

    # NOx calculation
    for i in range(5, ws_new.max_row): active.append([ f"=SUM('10 in 1'!{get_column_letter(2*j+2)}{i}:{get_column_letter(2*j+3)}{i})" for j in range(10)])
    print(f'NOx calculated after {time.time()-start:.2f} sec')
    
    align(ws_new); auto_fit(ws_new); print(f"new xlsx formatted after {time.time()-start:.2f} sec")
    wb_new.save(f"{datetime.today().month}月 last {day} day NOx Clipboard.xlsx")

def NOx_transfer(wb_orig, ws_new, old_sheets, old_rows, once_column: int, repeated_columns: list):
    for sheet_idx, sheet in enumerate(wb_orig.sheetnames[old_sheets[0]: old_sheets[1]]):
        source_ws = wb_orig[sheet]
        if sheet_idx == 0: # col A, O, P
            time = np.array([[cell for cell in row] for row in source_ws.iter_rows(min_row=old_rows[0], max_row=old_rows[1], min_col=once_column, max_col=once_column, values_only=True)])
        for column in repeated_columns:
            values = np.array([[cell for cell in row] for row in source_ws.iter_rows(min_row=old_rows[0], max_row=old_rows[1], min_col=column[0], max_col=column[1], values_only=True)])
            time = np.concatenate((time, values), axis=1)
    # print(time.shape, time[0]) # Debug
    for row in time:
        ws_new.append(list(row))

In [4]:
def paste(source_ws, source_range, target_ws, deviation, Fill):
    global count
    min_col, min_row, max_col, max_row = range_boundaries(source_range); count += (max_row-min_row+1)
    max_col, max_row = min(max_col, source_ws.max_column), min(max_row, source_ws.max_row) # dodge overfitting range
    # print(f'target boundaries: ({min_col},{min_row}), ({max_col}, {max_row})') # debug

    values = [[cell for cell in row] for row in source_ws.iter_rows(min_row=min_row, max_row=max_row, min_col=min_col, max_col=max_col, values_only=True)] # ;print(values[0], "\n...\n", values[-1]) # debugging
    for row_idx, row in enumerate(range(min_row, max_row+1)):
        for col_idx, col in enumerate(range(min_col, max_col + 1)):
            # print(row_idx, row, col_idx, col) # debug
            cell = target_ws.cell(row=row + deviation[1], column=col + deviation[0]+1)
            cell.value, cell.fill  = values[row_idx][col_idx], Fill

def value_copy(source_file, old_sheet, source_ranges, target_file, new_sheet, deviations, color = 'FFFFCC', keep_formula = True):
    start = time.time(); Fill = PatternFill(start_color=color, fill_type="solid")
    source_ws = load_workbook(source_file, data_only = True, read_only= True)[old_sheet]; print(f'{source_file} loaded after {time.time()-start:.2f} sec')
    target_wb = load_workbook(target_file, data_only=(not keep_formula)); target_ws = target_wb[new_sheet]; print(f'{target_file} loaded after {time.time()-start:.2f} sec')
    global count; count = 0
    
    for source_range, deviation in tqdm(zip(source_ranges, deviations), total=len(source_ranges), colour="#88FF88", 
                                        bar_format='{n_fmt}/{total_fmt} source ranges', ascii="- █"):
        paste(source_ws, source_range, target_ws, deviation, Fill)# ; print(source_range, deviation) # debug
        
    target_wb.save(target_file); print(f'{count} data transferred after {time.time()-start:.2f} sec\n')

### Call

- NOx Clipboard
- Excel transfer

In [5]:
NOx_clipboard(filepath = "BPPS_NOx last Fri - Thur (1h).xlsm", day = 7)

source xlsx loaded after 0.07 sec
headers transferred after 0.10 sec
NO, NO2 transferred after 0.32 sec
NOx calculated after 0.32 sec
new xlsx formatted after 0.48 sec


In [ ]:
test = 3
month = 'Mar'

# Step 1: 1182 => Supplecmentary
for i in range(1,test):
    # C_ paste, e.g. from 1182 Mar C1.xlsx
    i = 2 # focus for testing
    value_copy(f'1182 {month} C{i}.xlsx', "1182_Blended_Natural_Gas_Fuel_P", 
               ['M10:M753', 'N10:N753', 'O10:O753', 'P10:P753', 'Z10:Z753', 'AA10:AA753', 'AB10:AB753', 'AC10:AC753', 'AM10:AM753', 'AN10:AN753', 'AO10:AO753', 'AP10:AP753'], 
               'Supplementary Info to EPD(Mar 2025) - 複製.xlsx', f"C{i}", 
               [(-12, -7), (-10, -7), (-5, -7), (-9, -7), (-24, -7), (-22, -7), (-17, -7), (-21, -7), (-36, -7), (-34, -7), (-29, -7), (-33, -7)], 
               keep_formula=False)

### Test

In [ ]:
= MONTH(IF(WEEKDAY(TODAY()+1)=7, TODAY(), TODAY()-WEEKDAY(TODAY()+1))-7)
= DAY(IF(WEEKDAY(TODAY()+1)=7, TODAY(), TODAY()-WEEKDAY(TODAY()+1))-7)
###
= MONTH(IF(WEEKDAY(TODAY()+1)=7, TODAY(), TODAY()-WEEKDAY(TODAY()+1)))
= DAY(IF(WEEKDAY(TODAY()+1)=7, TODAY(), TODAY()-WEEKDAY(TODAY()+1))-1)
###
=TEXT(TODAY()-WEEKDAY(TODAY(),16),"d/m/yy")& "-" & TEXT(TODAY()-WEEKDAY(TODAY(),16)+6,"d/m/yy")